1. Configuração

In [1]:
import json
import sys
import io
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Cria diretório de artefatos se não existir
Path("artifacts").mkdir(exist_ok=True)

In [2]:
# Executando a ingestão
# https://ggplot2.tidyverse.org/reference/diamonds.html
# https://www.han.nl/opleidingen/master/applied-data-science/deeltijd/Introduction-to-Data-Science.pdf
df_raw = sns.load_dataset("diamonds")
print(f"Dimensões da base: {df_raw.shape}")
df_raw.head(3)

Dimensões da base: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31


2. Validação

In [33]:
def validate(df: pd.DataFrame) -> pd.DataFrame:
    """Valida tipos, nulos críticos e faixas de valores (modo informativo)."""
    expected_columns = [
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "x",
        "y",
        "z",
    ]

    # 1. Verifica colunas obrigatórias
    missing_cols = [col for col in expected_columns if col not in df.columns]
    if missing_cols:
        print(f"[ALERTA] Colunas ausentes no dataset: {missing_cols}")

    # Garante que só verificaremos as colunas que realmente existem no DataFrame
    key_cols = [col for col in ["carat", "x", "y", "z"] if col in df.columns]

    # 2. Verifica valores nulos em colunas-chave
    if key_cols and df[key_cols].isnull().sum().sum() > 0:
        print("[ALERTA] Existem valores nulos em colunas numéricas essenciais.")

    # 3. Valida faixas físicas (ex: dimensões e preço não podem ser <= 0)
    if key_cols and (df[key_cols] <= 0).any().any():
        print("[ALERTA] Encontrados valores menores ou iguais a zero em colunas físicas ou preço.")
        print(df.describe())

    if not missing_cols and key_cols and df[key_cols].isnull().sum().sum() == 0 and not (df[key_cols] <= 0).any().any():
        print("Validação concluída com sucesso: Schema e integridade física OK.")
        
    return df

In [34]:
df_validated = validate(df_raw)

[ALERTA] Encontrados valores menores ou iguais a zero em colunas físicas ou preço.
              carat         depth         table         price             x  \
count  53940.000000  53940.000000  53940.000000  53940.000000  53940.000000   
mean       0.797940     61.749405     57.457184   3932.799722      5.731157   
std        0.474011      1.432621      2.234491   3989.439738      1.121761   
min        0.200000     43.000000     43.000000    326.000000      0.000000   
25%        0.400000     61.000000     56.000000    950.000000      4.710000   
50%        0.700000     61.800000     57.000000   2401.000000      5.700000   
75%        1.040000     62.500000     59.000000   5324.250000      6.540000   
max        5.010000     79.000000     95.000000  18823.000000     10.740000   

                  y             z  
count  53940.000000  53940.000000  
mean       5.734526      3.538734  
std        1.142135      0.705699  
min        0.000000      0.000000  
25%        4.720000      

In [5]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
  """Remove duplicatas e filtra inconsistências severas."""
  df_clean = df.copy()

  # Remove linhas duplicadas exatas
  initial_len = len(df_clean)
  df_clean = df_clean.drop_duplicates()
  print(f"Removidas {initial_len - len(df_clean)} linhas duplicadas.")

  # Remove diamantes com dimensões físicas zeradas (ex: x=0 ou y=0)
  df_clean = df_clean[(df_clean["x"] > 0) & (df_clean["y"] > 0)]

  return df_clean

In [6]:
df_cleaned = clean(df_validated)

Removidas 146 linhas duplicadas.


3. Tratamento das variáveis

In [8]:
def features(df: pd.DataFrame):
  """Cria novas features e separa matriz X e vetor alvo y."""
  df_feat = df.copy()

  # Feature engineering: Volume aproximado do diamante (elipsóide/paralelepípedo)
  df_feat["volume"] = df_feat["x"] * df_feat["y"] * df_feat["z"]

  # Remove colunas originais redundantes se necessário, ou mantém x, y, z
  X = df_feat.drop(columns=["price"])
  if "price" in df_feat.columns:
        X = df_feat.drop(columns=["price"])
        y = df_feat["price"]
        return X, y
  else:
        # Na produção, não temos o preço real
        return df_feat, None

In [9]:
X, y = features(df_cleaned)

5. Treinamento

In [10]:
def train_baseline(X_train, y_train, preprocessor):
    """Treina o modelo Baseline (Dummy Regressor) para comparação."""
    baseline_pipe = Pipeline(
        steps=[
            ("prep", preprocessor), 
            ("model", DummyRegressor(strategy="mean"))
        ]
    )
    baseline_pipe.fit(X_train, y_train)
    print("Baseline treinado com sucesso.")
    return baseline_pipe

In [11]:
def train_random_forest(X_train, y_train, preprocessor, seed=42):
    """Otimiza hiperparâmetros via Random Search e treina o Random Forest."""
    
    # 1. Pipeline base
    rf_pipe = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", RandomForestRegressor(random_state=seed, n_jobs=-1)),
        ]
    )

    # 2. Espaço de busca de hiperparâmetros
    param_distributions = {
        "model__n_estimators": [30, 50, 100, 200],
        "model__max_depth": [10, 15, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    }

    # 3. Configuração do RandomizedSearchCV com validação cruzada[cite: 5]
    random_search = RandomizedSearchCV(
        estimator=rf_pipe,
        param_distributions=param_distributions,
        n_iter=5,          
        cv=3,              
        scoring='neg_mean_squared_error',      
        random_state=seed,
        n_jobs=-1
    )

    print("Executando Random Search para otimização do Random Forest...")
    random_search.fit(X_train, y_train)

    print(f"Melhores parâmetros encontrados: {random_search.best_params_}")
    
    # Retorna o melhor estimador já ajustado com todos os dados de treino
    return random_search.best_estimator_

In [12]:
def train(X, y, test_size=0.3, seed=42):
    """Função orquestradora da etapa de treino: divide os dados e chama os modelos."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )

    # Identificando colunas numéricas e categóricas
    numeric_features = ["carat", "depth", "table", "x", "y", "z", "volume"]
    categorical_features = ["cut", "color", "clarity"]

    # Pré-processador unificado (evita data leakage)[cite: 5]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
    
    # Chamando as funções separadas para cada modelo
    baseline_pipe = train_baseline(X_train, y_train, preprocessor)
    best_model = train_random_forest(X_train, y_train, preprocessor, seed=seed)

    print("Treinamento e otimização concluídos com sucesso.")
    return best_model, baseline_pipe, X_test, y_test

In [13]:
model_rf, baseline, X_test, y_test = train(X, y, test_size=0.3, seed=42)

Baseline treinado com sucesso.
Executando Random Search para otimização do Random Forest...
Melhores parâmetros encontrados: {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_depth': 20}
Treinamento e otimização concluídos com sucesso.


6. Avaliação dos Modelos

In [16]:
def evaluate(model, baseline, X_test, y_test) -> dict:
  """Calcula métricas de desempenho comparando com o baseline."""
  preds = model.predict(X_test)
  base_preds = baseline.predict(X_test)

  metrics = {
      "mae": float(mean_absolute_error(y_test, preds)),
      "rmse": float(np.sqrt(mean_squared_error(y_test, preds))),
      "r2": float(r2_score(y_test, preds)),
      "baseline_mae": float(mean_absolute_error(y_test, base_preds)),
      "baseline_r2": float(r2_score(y_test, base_preds)),
  }

  print(f"--- Avaliação do Modelo ---")
  print(f"MAE: {metrics['mae']:.2f} (Baseline MAE: {metrics['baseline_mae']:.2f})")
  print(f"RMSE: {metrics['rmse']:.2f}")

  return metrics

In [17]:
metrics = evaluate(model_rf, baseline, X_test, y_test)

--- Avaliação do Modelo ---
MAE: 273.21 (Baseline MAE: 2987.67)
RMSE: 549.80


7. Salvamento de Artefatos

In [18]:
def save_artifacts(model, metrics: dict, version="v1", logs_texto: str = ""):
    """Salva o pipeline, metadados, relatório em Markdown e os logs de execução."""
    
    # 1. Salva o pipeline (pré-processador + modelo)
    model_path = f"artifacts/model_{version}.joblib"
    joblib.dump(model, model_path)

    # 2. Extrai de forma segura apenas os nomes das etapas/transformadores para JSON
    transformers_info = []
    # Tratamento caso o modelo seja um GridSearchCV/RandomizedSearchCV ou Pipeline direto
    estimator = model.best_estimator_ if hasattr(model, "best_estimator_") else model
    
    for name, trans, cols in estimator.named_steps["prep"].transformers_:
        transformers_info.append({
            "name": name,
            "transformer": type(trans).__name__,  # Nome da classe (ex: 'StandardScaler')
            "columns": cols
        })

    model_type = type(estimator.named_steps["model"]).__name__

    # 3. Salva os metadados
    metadata = {
        "version": version,
        "model_type": model_type,
        "transformers": transformers_info,
        "metrics": metrics,
    }
    
    with open(f"artifacts/metadata_{version}.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

    print(f"Artefatos salvos com sucesso na pasta 'artifacts/' para a versão {version} (incluindo modelo e metadados).")

In [19]:
save_artifacts(model_rf, metrics, version="v1")

Artefatos salvos com sucesso na pasta 'artifacts/' para a versão v1 (incluindo modelo e metadados).


8. Função Orquestradora

In [22]:
def run_pipeline():
  print("=== INÍCIO DO PIPELINE ===")
  # 1. Ingestão
  df = sns.load_dataset("diamonds")
  # 2. Validação
  df = validate(df)
  # 3. Limpeza
  df = clean(df)
  # 4. Features
  X, y = features(df)
  # 5. Treino
  model, baseline, X_test, y_test = train(X, y)
  # 6. Avaliação
  metrics = evaluate(model, baseline, X_test, y_test)
  # 7. Salvar Artefatos
  save_artifacts(model, metrics, version="v1")
  print("=== PIPELINE EXECUTADO COM SUCESSO ===")
  return metrics

In [21]:
# Executando o pipeline completo
metrics_result = run_pipeline()

=== INÍCIO DO PIPELINE ===
[ALERTA] Encontrados valores menores ou iguais a zero em colunas físicas ou preço.
              carat         depth         table         price             x  \
count  53940.000000  53940.000000  53940.000000  53940.000000  53940.000000   
mean       0.797940     61.749405     57.457184   3932.799722      5.731157   
std        0.474011      1.432621      2.234491   3989.439738      1.121761   
min        0.200000     43.000000     43.000000    326.000000      0.000000   
25%        0.400000     61.000000     56.000000    950.000000      4.710000   
50%        0.700000     61.800000     57.000000   2401.000000      5.700000   
75%        1.040000     62.500000     59.000000   5324.250000      6.540000   
max        5.010000     79.000000     95.000000  18823.000000     10.740000   

                  y             z  
count  53940.000000  53940.000000  
mean       5.734526      3.538734  
std        1.142135      0.705699  
min        0.000000      0.000000 

# Pipeline de produção

In [38]:
def run_inference(new_data_source: str, version="v1"):
    print("=== INÍCIO DO PIPELINE DE PRODUÇÃO (INFERÊNCIA) ===")
    
    # 1. Carrega o artefato salvo no treino (Pipeline completo: pré-processador + modelo)
    model_path = f"artifacts/model_{version}.joblib"
    print(f"Carregando modelo treinado de: {model_path}")
    pipeline = joblib.load(model_path)
    
    # 2. Ingestão de novos dados (ex: novos diamantes que chegaram para avaliação)
    df_new = new_data_source.copy()
    
    # 3. Validação
    df_new = validate(df_new)
    
    # 4. Limpeza
    df_new = clean(df_new)
    
    # 5. Features
    X_new, _ = features(df_new)
    
    # 6. Predição 
    print("Gerando predições...")
    predictions = pipeline.predict(X_new)
    
    # 7. Entrega / Salvamento dos Resultados
    df_new["preco_previsto"] = predictions
    
    output_path = f"artifacts/predictions_{version}.csv"
    df_new.to_csv(output_path, index=False, encoding="utf-8")
    
    print(f"Predições geradas e salvas com sucesso em '{output_path}'.")
    print("=== FIM DO PIPELINE DE PRODUÇÃO ===")
    
    return df_new.head()

In [39]:
dados = {
    'carat': [0.23],
    'cut': ['Ideal'],
    'color': ['E'],
    'clarity': ['SI2'],
    'depth': [61.5],
    'table': [55.0],
    'price': [326],
    'x': [3.95],
    'y': [3.98],
    'z': [2.43]
}
df = pd.DataFrame(dados)
display(df)

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43


In [40]:
# --- Executando a Produção ---
df_resultados = run_inference(df, version="v1")
display(df_resultados)

=== INÍCIO DO PIPELINE DE PRODUÇÃO (INFERÊNCIA) ===
Carregando modelo treinado de: artifacts/model_v1.joblib
Validação concluída com sucesso: Schema e integridade física OK.
Removidas 0 linhas duplicadas.
Gerando predições...
Predições geradas e salvas com sucesso em 'artifacts/predictions_v1.csv'.
=== FIM DO PIPELINE DE PRODUÇÃO ===


,carat,cut,color,clarity,depth,table,price,x,y,z,preco_previsto
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43,422.187998


# Atividade:
1. Adicione um modelo LASSO ao pipeline.
2. Ele tem métrica de desempenho melhor do que o Random Forest? 